# Gemini-2.5-Flash Response Collection
## Thesis Phase II — standalone collection notebook
Collects responses for all 15 questions × 2 languages = 30 queries.
Output: `gemini_responses.json` in your evaluation folder.
Merge into main notebook using the merge cell in `chatbot_eval.ipynb`.

In [ ]:
!pip install -q google-genai tqdm
print("✅ Done")

✅ Done


In [ ]:
import os, json, time
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
print("Imports OK ✅")

Imports OK ✅


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
BASE     = Path("/content/drive/MyDrive/medical_protocols")
OUT_DIR  = BASE / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
QUESTIONS_JSON = BASE / "question_design_v1" / "final_questions.json"
print(f"OUT_DIR: {OUT_DIR}")

OUT_DIR: /content/drive/MyDrive/medical_protocols/evaluation


In [ ]:

GEMINI_API_KEY = ""   # ← paste key here, e.g. "AIzaSy..."

if not GEMINI_API_KEY:
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        pass

print("GEMINI_API_KEY:", "✅ set" if GEMINI_API_KEY else "❌ MISSING")

GEMINI_API_KEY: ✅ set


In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical assistant specialising in obstetrics and gynaecology. "
    "Answer the following question based on evidence-based clinical guidelines. "
    "Be specific and include: diagnostic criteria, medication names and dosages "
    "where relevant, gestational age thresholds, contraindications, and "
    "recommended follow-up intervals. "
    "Do not give general health advice — provide protocol-level clinical detail."
)

print("System prompt:")
print("-" * 60)
print(SYSTEM_PROMPT)
print("-" * 60)
print(f"Length: {len(SYSTEM_PROMPT)} characters")

System prompt:
------------------------------------------------------------
You are a clinical assistant specialising in obstetrics and gynaecology. Answer the following question based on evidence-based clinical guidelines. Be specific and include: diagnostic criteria, medication names and dosages where relevant, gestational age thresholds, contraindications, and recommended follow-up intervals. Do not give general health advice — provide protocol-level clinical detail.
------------------------------------------------------------
Length: 398 characters


In [ ]:
questions = json.loads(QUESTIONS_JSON.read_text(encoding="utf-8"))
print(f"Loaded {len(questions)} questions")
for q in questions: print(f"  {q['id']} | EN: {q['question_en'][:60]}...")

Loaded 15 questions
  A1 | EN: What are the diagnostic criteria for severe preeclampsia, an...
  A2 | EN: What fasting glucose and oral glucose tolerance test thresho...
  A3 | EN: What clinical and laboratory criteria define septic shock in...
  B1 | EN: What is the recommended first-line antihypertensive therapy ...
  B2 | EN: What empirical antibiotic regimen is recommended for obstetr...
  B3 | EN: What tocolytic agents are recommended for preterm labour, an...
  C1 | EN: What are the stepwise surgical interventions recommended for...
  C2 | EN: What are the indications for emergency hysterectomy in obste...
  C3 | EN: What are the clinical signs and immediate management steps f...
  D1 | EN: What antenatal screening tests are recommended during the fi...
  D2 | EN: What glucose monitoring targets are recommended for patients...
  D3 | EN: What risk factors identify pregnant women who should receive...
  E1 | EN: What are the diagnostic criteria for chorioamnionitis and wh...
  E2 

In [ ]:
def query_gemini(question: str, system_prompt: str,
                 max_retries: int = 6, base_delay: float = 15.0) -> str:
    """Query Gemini-2.5-Flash with exponential backoff for 503/429 errors."""
    from google import genai as google_genai
    from google.genai import types as genai_types
    client = google_genai.Client(api_key=GEMINI_API_KEY)
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=question,
                config=genai_types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    temperature=0,
                    max_output_tokens=1024,
                ),
            )
            return resp.text.strip()
        except Exception as e:
            err_str = str(e)
            is_retryable = any(x in err_str for x in
                               ["503", "429", "UNAVAILABLE", "RESOURCE_EXHAUSTED"])
            if is_retryable and attempt < max_retries - 1:
                wait = base_delay * (2 ** attempt)  # 15, 30, 60, 120, 240, 480s
                print(f"  [{attempt+1}/{max_retries}] Rate limited — waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                raise

print("query_gemini() ready ✅")

query_gemini() ready ✅


In [ ]:
SAVE_PATH = OUT_DIR / "gemini_responses.json"

if SAVE_PATH.exists():
    collected = json.loads(SAVE_PATH.read_text(encoding="utf-8"))
    print(f"Resumed: {len(collected)} existing responses found.")
else:
    collected = []
    print("Starting fresh.")

done_keys = {(r["question_id"], r["language"]) for r in collected}
pending = [(q, lang) for q in questions for lang in ["EN", "RU"]
           if (q["id"], lang) not in done_keys]

print(f"Pending: {len(pending)} / 30 queries")

for q, lang in tqdm(pending, desc="Gemini-2.5-Flash"):
    q_text = q["question_en"] if lang == "EN" else q["question_ru"]
    try:
        response_text = query_gemini(q_text, SYSTEM_PROMPT)
    except Exception as e:
        response_text = f"ERROR: {e}"
        print(f"  ❌ {q['id']} {lang}: {e}")

    collected.append({
        "bot":             "Gemini-2.5-Flash",
        "question_id":     q["id"],
        "stratum":         q.get("stratum", ""),
        "language":        lang,
        "question_text":   q_text,
        "response_text":   response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":       datetime.utcnow().isoformat(),
    })
    SAVE_PATH.write_text(json.dumps(collected, ensure_ascii=False, indent=2), encoding="utf-8")
    time.sleep(5.0)  # 5s between calls to stay under rate limit

print(f"\n✅ Done! {len(collected)}/30 responses saved to:")
print(f"   {SAVE_PATH}")

errors = [r for r in collected if r["response_text"].startswith("ERROR:")]
print(f"Errors: {len(errors)} / {len(collected)}")
if errors:
    for r in errors: print(f"  ❌ {r['question_id']} {r['language']}: {r['response_text'][:80]}")
else:
    print("All responses clean ✅")

Starting fresh.
Pending: 30 / 30 queries


Gemini-2.5-Flash:   0%|          | 0/30 [00:00<?, ?it/s]/tmp/ipykernel_331/4018384051.py:33: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":       datetime.utcnow().isoformat(),
Gemini-2.5-Flash:  27%|██▋       | 8/30 [01:48<04:35, 12.52s/it]

  [1/6] Rate limited — waiting 15s...
  [2/6] Rate limited — waiting 30s...
  [3/6] Rate limited — waiting 60s...
  [4/6] Rate limited — waiting 120s...
  [5/6] Rate limited — waiting 240s...
  ❌ B2 EN: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 52.962305068s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguag

Gemini-2.5-Flash:  30%|███       | 9/30 [09:42<54:55, 156.92s/it]

  [1/6] Rate limited — waiting 15s...
  [2/6] Rate limited — waiting 30s...
  [3/6] Rate limited — waiting 60s...
  [4/6] Rate limited — waiting 120s...
  [5/6] Rate limited — waiting 240s...
  ❌ B2 RU: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 57.837008768s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguag

...

  [1/6] Rate limited — waiting 15s...
  [2/6] Rate limited — waiting 30s...
  [3/6] Rate limited — waiting 60s...
  [4/6] Rate limited — waiting 120s...
  [5/6] Rate limited — waiting 240s...
  ❌ C1 EN: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 19.395181037s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguag

Gemini-2.5-Flash:  43%|████▎     | 13/30 [41:16<1:53:12, 399.56s/it]

  [1/6] Rate limited — waiting 15s...
  [2/6] Rate limited — waiting 30s...
  [3/6] Rate limited — waiting 60s...
  [4/6] Rate limited — waiting 120s...
  [5/6] Rate limited — waiting 240s...


Gemini-2.5-Flash:  43%|████▎     | 13/30 [48:16<1:03:07, 222.77s/it]


KeyboardInterrupt: 